In [9]:
from pathlib import Path
import sqlite3
import pandas as pd
import numpy as np
import json
import ast
from tqdm import tqdm
from multiprocessing import Pool, cpu_count

from forge_net.data.dataloaders import GetSingleStepDataLoaders
from forge_net.utils.common import actions_from_feature_map
from forge_net.utils.common import MeshContainer, meshcontainer_to_volume
from forge_net.utils.math import *
import meshio
pv.start_xvfb()

In [10]:
def process_series(args):
    """Process a single series - this will run in parallel
    
    Args:
        args: Tuple of (series_id, group_df, total_points, press_width, mask_points, seed)
    """
    series_id, group_df, total_points, press_width, mask_points, seed = args
    series_coords_t = []
    series_coords_tp1 = []
    series_steps = []
    series_positions = []
    series_rotations = []
    pv_meshes = []
    pv_meshes_tp1 = []
    bary_coords_list = []
    tri_ids_list = []
    
    for i in range(len(group_df) - 1):
        row_t = group_df.iloc[i]
        row_tp1 = group_df.iloc[i + 1]
       

        # Input mesh (coords from frame i)
        num_steps_t = row_t["solver_steps"]
        vertices_t = np.array(ast.literal_eval(row_t["vertices"])).reshape(-1,3)
        triangles_t = np.array(ast.literal_eval(row_t["triangles"])).reshape(-1,4)
        vertex_temps_t = np.array(ast.literal_eval(row_t["input_temperature"])).reshape(-1,1)
        print(np.array(vertices_t).reshape(-1,3).shape, np.array(triangles_t).reshape(-1,4).shape)

        tmp_mesh_t = meshio.Mesh(
            points=vertices_t, 
            cells=[("tetra", triangles_t)]
        )
        pv_mesh_t = pv.from_meshio(tmp_mesh_t)
        # pl = pv.Plotter()
        # pl.add_mesh(pv_mesh_t)
        # pl.screenshot("tmp.png")

        # print(tmp_mesh_t)
        # print(type(pv_mesh_t)) 

        num_steps_tp1 = row_tp1["solver_steps"]
        vertices_tp1 = np.array(ast.literal_eval(row_tp1["vertices"])).reshape(-1,3)
        triangles_tp1 = np.array(ast.literal_eval(row_tp1["triangles"])).reshape(-1,4)
        vertex_temps_tp1 = np.array(ast.literal_eval(row_tp1["input_temperature"])).reshape(-1,1)
        tmp_mesh_tp1 = meshio.Mesh(
            points=vertices_tp1, 
            cells=[("tetra", triangles_tp1)]
        )
        pv_mesh_tp1 = pv.from_meshio(tmp_mesh_tp1)
 
        # p_tp1 = ((row_tp1["x_max_band"] + row_tp1["x_min_band"]) / 2, 0 , 0)
        # r_tp1 = eulerxyz_to_quat((row_tp1["rotation_euler_x"], 0, 0)) #-> quaternion
        # pv_mesh_t.points = transform_points(np.array(pv_mesh_t.points), np.array(r_tp1), np.array(p_tp1))
        # pv_mesh_tp1.points = transform_points(np.array(pv_mesh_tp1.points), np.array(r_tp1), np.array(p_tp1))

        sampled_points_t, point_triangle_ids, bary_coords, sampled_temps_t = tetrahedral_barycentric_sampling(
                                                                                                            pv_mesh_t, 
                                                                                                            total_points, 
                                                                                                            node_features=vertex_temps_t, 
                                                                                                            seed=seed)
        print("sampled barycenters:" , sampled_points_t)
        tri_ids_list.append(point_triangle_ids)
        bary_coords_list.append(bary_coords)

        sampled_points_tp1, sampled_temps_tp1 = update_tetrahedral_barycentric_points(
                                                                                deformed_mesh=pv_mesh_tp1, 
                                                                                tet_ids=point_triangle_ids, 
                                                                                barycentric_coords=bary_coords, 
                                                                                node_features=vertex_temps_tp1)
        
        pl = pv.Plotter()

        pv_mesh_tp1.point_data["Temperature"] = vertex_temps_tp1.flatten()
        pl.add_mesh(
            pv_mesh_tp1, 
            scalars="Temperature",  # Tell PyVista to color by this attribute
            cmap="coolwarm",         # A great colormap for temperature (Blue to Red)
            show_scalar_bar=True     # Displays the color legend
        )
        pl.show_grid()
        pl.screenshot("tmp_mesh.png")

        pl = pv.Plotter()
        # pl.add_mesh(pv_mesh_tp1,style='wireframe')
        point_cloud = pv.PolyData(sampled_points_tp1)
        point_cloud["temps"] = sampled_temps_t
        pl.add_mesh(point_cloud,
                    scalars='temps',
                    point_size=5.0,render_points_as_spheres=True)
        pl.show_grid()
        pl.screenshot("tmp.png")
        # pl.export_html("tmp.html")

        pl = pv.Plotter()
        # pl.add_mesh(pv_mesh_tp1,style='wireframe')
        point_cloud = pv.PolyData(sampled_points_tp1)
        point_cloud["temps"] = sampled_temps_tp1
        pl.add_mesh(point_cloud,
                    scalars='temps',
                    point_size=5.0,render_points_as_spheres=True)
        pl.show_grid()
        pl.screenshot("tmp_tp1.png")
        # pl.export_html("tmp_tp1.html")


        

In [11]:
# series_id, group_df, total_points, press_width, mask_points, seed = args
db_path="/local/scratch/groves/jax-forgeRL/JAX-FORGE/Agility_Forge_data/data/forge_database.db"
lines=1_000
total_points=10_000
mask_points=False
seed=None
conn = sqlite3.connect(db_path)
df = pd.read_sql_query(f"SELECT * FROM hits LIMIT {int(lines)};", conn)
conn.close()

# Prepare arguments for each series
press_width = 1.0
series_ids = df['series_id'].unique()
args_list = [
    (series_id, 
        df[df['series_id'] == series_id].reset_index(drop=True), 
        total_points, 
        press_width,
        mask_points,
        seed
    )
    for series_id in series_ids
]

In [12]:
process_series(args_list[0])

(7531, 3) (30790, 4)
sampled barycenters: [[ 7.48667821  1.31182709 -3.10942878]
 [ 9.66416053  4.73910448  2.91137344]
 [ 5.14325067  2.38946175  1.11022477]
 ...
 [-2.93614387 -3.32067912  1.97251846]
 [ 6.04782639 -7.32347373 -1.76695468]
 [64.3919589   6.37115358  2.2236064 ]]
(7531, 3) (30790, 4)
sampled barycenters: [[ 8.49634213  1.16527724 -3.74781485]
 [ 9.60429925  3.73644042  2.81240377]
 [ 4.80393502  2.40704399  0.92939046]
 ...
 [38.47260074 -2.97986959  4.14933421]
 [47.60875699 -2.72223083 -7.52301151]
 [-2.58467672 -2.03852678 -7.59191507]]
(7531, 3) (30790, 4)
sampled barycenters: [[ 8.68706585  0.42700701 -3.00764744]
 [ 9.02491576  3.31819136  2.92862173]
 [ 4.8524871   3.95300667  1.91134949]
 ...
 [16.71428255 -5.69327238  1.70289714]
 [82.56400804  0.414979    8.38092465]
 [87.91464535 -0.64046682 -7.19330844]]
(7531, 3) (30790, 4)
sampled barycenters: [[ 8.37842882  1.82705969 -2.95212202]
 [ 9.35123959  4.73938756  2.00777712]
 [ 4.97142349  3.19985014 -0.33647